# f_b prediction — Stage 1: controlled sim validation

The masked observable model (`fm_observables_masked`) can condition on a **subset** of the R200 observables and predict the rest. Here we turn it into a **baryon-fraction predictor**: condition on what a survey delivers (headline = lensing `M` + SZ `Y` + X-ray `Tx`), **withhold** the baryon-mass observables, and read f_b = (Mgas+Mstar)/M200 off the *generated* field — a genuine prediction since its numerator was never an input.

This notebook is the controlled test on held-out CAMELS-TNG halos (truth available) that must pass before the real eROSITA-group application is interpretable:
1. **Subset ablation** — f_b bias/scatter vs truth for `{M}`, `{M,Y}`, `{M,Tx}`, `{M,Y,Tx}`, `{M,Mstar}`, full.
2. **Mass trend** — predicted vs truth f_b–M.
3. **Feedback recovery** — does predicted f_b track the SN/AGN feedback parameters the way truth does (physics learned from observables alone)?
4. **DMO-swap viability** — predict f_b *without* the halo's own DMO (matched-mass templates), the sim proxy for real data + its uncertainty; plus coverage/calibration.
5. **f_b(<r) profile** — the radial baryon fraction (r≠R200 is not conditioned).

Scope: CAMELS groups, logM200 ~ 12.7–14. Kernel: **torch3** (GPU). Env: `OBS_RUN_DIR`, `OBS_CKPT`, `N_HALOS`.

In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, '/mnt/home/mlee1/vdm_bind2/examples')   # fb_predict helpers
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path
from bind.data import (load_file_list, AstroDataset, NormStats, OBSERVABLE_KEYS, N_OBS,
                       THERMO_KEYS, r200_from_sample)
from bind.params import PARAM_NAMES
from bind.train import FlowMatchingLit
import fb_predict as fbp

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_DIR   = Path(os.environ.get('OBS_RUN_DIR', '/mnt/home/mlee1/ceph/fm_runs/fm_observables_masked'))
CKPT      = os.environ.get('OBS_CKPT', 'last.ckpt')
DATA_ROOT = Path(os.environ.get('BIND_DATA_ROOT', '/mnt/home/mlee1/ceph/train_data_rotated2_128_cpu'))
N_HALOS   = int(os.environ.get('N_HALOS', 256))
N_STEPS, BATCH = 50, 32
HEADLINE  = 'M+Y+Tx'

ns = NormStats.load(RUN_DIR / 'norm_stats.npz')
assert ns.condition_observables and ns.mask_observables, 'need a --mask_observables run'
ns.mask_observables = False    # deterministic full-obs reads; we pack subset masks via fb_predict
model = FlowMatchingLit.load_from_checkpoint(str(RUN_DIR / 'checkpoints' / CKPT),
                                             map_location=device).eval().to(device)
fm = model.fm
print(f'loaded {CKPT}  n_params={model.hparams.n_params} (expect {2*N_OBS})')

In [ ]:
files = load_file_list(str(DATA_ROOT), 'test')
ds = AstroDataset(files, ns)
rng = np.random.default_rng(0)
idx = rng.choice(len(ds), size=min(N_HALOS, len(ds)), replace=False)

conds, lss, obs_norm, truth_fb, logM, r200s, params = [], [], [], [], [], [], []
for i in idx:
    it = ds[i]; d = np.load(files[i])
    M, r2 = float(d['halo_mass']), r200_from_sample(d)
    conds.append(it['condition']); lss.append(it['large_scale'])
    obs_norm.append(it['params'].numpy())                          # normalized N_OBS truth observables
    truth_maps = np.concatenate([d['target'], np.stack([d[k] for k in THERMO_KEYS])])
    truth_fb.append(fbp.aperture_fb(truth_maps, M, r2))
    logM.append(np.log10(M)); r200s.append(r2); params.append(d['params'])
conds, lss = torch.stack(conds), torch.stack(lss)
obs_norm = np.stack(obs_norm); truth_fb = np.array(truth_fb)
logM = np.array(logM); r200s = np.array(r200s); params = np.stack(params)
print(f'{len(idx)} halos | logM200 {logM.min():.2f}–{logM.max():.2f} | '
      f'truth f_b median {np.median(truth_fb):.3f}')

## 1. Subset ablation — which observables carry the f_b information?

Same DMO + same noise across subsets, so differences are purely the conditioning. `{M}` is the floor (DMO+mass only); each added observable should tighten f_b.

In [ ]:
SUBSETS = {
    'M':       ['M_200'],
    'M+Y':     ['M_200', 'Y_200'],
    'M+Tx':    ['M_200', 'Tx_200'],
    'M+Y+Tx':  ['M_200', 'Y_200', 'Tx_200'],
    'M+Mstar': ['M_200', 'Mstar_200'],
    'full':    list(OBSERVABLE_KEYS),
}
pred_fb = {}
M200 = 10.0 ** logM
for name, keys in SUBSETS.items():
    keep = fbp.subset_keep(keys)
    out = []
    for s in range(0, len(idx), BATCH):
        maps = fbp.generate(fm, ns, conds[s:s+BATCH], lss[s:s+BATCH], obs_norm[s:s+BATCH],
                            keep, device, N_STEPS, seed=100 + s)
        out += [fbp.aperture_fb(maps[k], M200[s+k], r200s[s+k]) for k in range(len(maps))]
    pred_fb[name] = np.array(out)

print(f'{"subset":9s} {"f_b bias":>9s} {"scatter":>9s} {"|frac err|":>10s}')
for name in SUBSETS:
    dd = pred_fb[name] - truth_fb
    fe = np.median(np.abs(dd) / truth_fb)
    print(f'{name:9s} {np.median(dd):+9.4f} {np.std(dd):9.4f} {fe:10.3f}')

In [ ]:
try:
    from scipy.stats import spearmanr
    sp = lambda a, b: spearmanr(a, b).statistic
except Exception:
    sp = lambda a, b: np.corrcoef(a, b)[0, 1]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
names = list(SUBSETS)
ax[0].bar(names, [np.std(pred_fb[n] - truth_fb) for n in names], color='steelblue')
ax[0].set_ylabel('f_b prediction scatter (dex-free)'); ax[0].set_title('Subset ablation')
ax[0].tick_params(axis='x', rotation=30)

p = pred_fb[HEADLINE]
sc = ax[1].scatter(truth_fb, p, c=logM, cmap='viridis', s=14)
lo, hi = np.percentile(np.r_[truth_fb, p], [1, 99]); ax[1].plot([lo, hi], [lo, hi], 'k--', lw=1)
ax[1].set_xlabel('truth f_b'); ax[1].set_ylabel(f'BIND f_b  ({HEADLINE})')
ax[1].set_title(f'{HEADLINE}:  bias {np.median(p-truth_fb):+.4f}, '
                f'scatter {np.std(p-truth_fb):.4f}, Spearman {sp(truth_fb, p):.2f}')
plt.colorbar(sc, ax=ax[1], label='log10 M200')
plt.tight_layout(); plt.show()

## 2. Mass trend — predicted vs truth f_b–M

In [ ]:
bins = np.linspace(logM.min(), logM.max(), 7)
bc = 0.5 * (bins[:-1] + bins[1:])
which = np.digitize(logM, bins) - 1
def binned(v):
    return np.array([np.median(v[which == b]) if (which == b).any() else np.nan for b in range(len(bc))])
fig, ax = plt.subplots(figsize=(6.5, 4.4))
ax.plot(bc, binned(truth_fb), 'k-o', label='truth')
ax.plot(bc, binned(pred_fb[HEADLINE]), 'r--s', label=f'BIND ({HEADLINE})')
ax.plot(bc, binned(pred_fb['M']), color='gray', ls=':', marker='^', label='M only (floor)')
ax.axhline(0.157, ls=':', color='C0', lw=1, label='cosmic \u03a9b/\u03a9m')
ax.set_xlabel('log10 M200'); ax.set_ylabel('f_b(<R200)'); ax.legend(fontsize=8)
ax.set_title('Baryon fraction–mass relation: recovery'); plt.tight_layout(); plt.show()

## 3. Feedback recovery — does f_b track the SN/AGN knobs?

Each held-out halo carries its sim's feedback parameters. If predicted f_b (from `M,Y,Tx` only) tracks the feedback parameters the way truth does, the observable→f_b mapping has captured the feedback physics. Drivers: `WindEnergyIn1e51erg` (SN) and `BlackHoleFeedbackFactor` (AGN).

In [ ]:
DRIVERS = ['WindEnergyIn1e51erg', 'BlackHoleFeedbackFactor']
fig, axes = plt.subplots(1, len(DRIVERS), figsize=(6*len(DRIVERS), 4.2))
for ax, pname in zip(np.atleast_1d(axes), DRIVERS):
    pj = PARAM_NAMES.index(pname); pv = params[:, pj]
    pb = np.linspace(pv.min(), pv.max(), 7); pc = 0.5 * (pb[:-1] + pb[1:])
    w = np.digitize(pv, pb) - 1
    tb = [np.median(truth_fb[w == b]) if (w == b).any() else np.nan for b in range(len(pc))]
    rb = [np.median(pred_fb[HEADLINE][w == b]) if (w == b).any() else np.nan for b in range(len(pc))]
    ax.plot(pc, tb, 'k-o', label='truth'); ax.plot(pc, rb, 'r--s', label=f'BIND ({HEADLINE})')
    ax.set_xlabel(pname); ax.set_ylabel('f_b(<R200)'); ax.legend(fontsize=8)
fig.suptitle('Feedback recovery: f_b vs feedback parameter'); plt.tight_layout(); plt.show()

## 4. DMO-swap viability + calibration (the real-data proxy)

Real groups have no DMO image, so we marginalize over matched-mass sim DMO templates. Here we test that on sims: predict each halo's f_b using **other** halos' DMO (±0.1 dex in logM), conditioned on its own `M,Y,Tx`. If the marginal prediction still recovers truth, the real-data path is viable; the template spread is the predictive uncertainty, and we check its coverage.

In [ ]:
keep = fbp.subset_keep(SUBSETS[HEADLINE])
n_eval = min(48, len(idx)); n_templates = 8
rs = np.random.default_rng(1)
swap_med, swap_lo, swap_hi, tru = [], [], [], []
for n in rs.choice(len(idx), n_eval, replace=False):
    pool = np.where((np.abs(logM - logM[n]) < 0.1) & (np.arange(len(idx)) != n))[0]
    if len(pool) < 3:
        continue
    fbs = fbp.predict_fb_marginal(fm, ns, obs_norm[n], keep, conds[pool], lss[pool],
                                  M200[n], r200s[n], device, n_templates=n_templates,
                                  n_steps=N_STEPS, seed0=int(n))
    swap_med.append(np.median(fbs)); swap_lo.append(np.percentile(fbs, 16))
    swap_hi.append(np.percentile(fbs, 84)); tru.append(truth_fb[n])
swap_med, swap_lo, swap_hi, tru = map(np.array, (swap_med, swap_lo, swap_hi, tru))

cover = np.mean((tru >= swap_lo) & (tru <= swap_hi))
fig, ax = plt.subplots(figsize=(6, 5))
ax.errorbar(tru, swap_med, yerr=[swap_med - swap_lo, swap_hi - swap_med], fmt='o', ms=4,
            color='C3', ecolor='gray', elinewidth=0.8, capsize=2)
lo, hi = np.percentile(np.r_[tru, swap_med], [1, 99]); ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
ax.set_xlabel('truth f_b'); ax.set_ylabel('BIND f_b (DMO-marginal)')
ax.set_title(f'DMO-swap: bias {np.median(swap_med-tru):+.4f}, '
             f'scatter {np.std(swap_med-tru):.4f}, 68%% coverage {cover:.2f}')
plt.tight_layout(); plt.show()
print(f'self-DMO scatter {np.std(pred_fb[HEADLINE]-truth_fb):.4f}  ->  '
      f'DMO-marginal scatter {np.std(swap_med-tru):.4f}  (extra cost of not knowing the DMO)')

## 5. f_b(<r) profile — the genuinely-predicted radial shape (r≠R200)

In [ ]:
radii = np.linspace(0.1, 2.5, 18)
nprof = min(48, len(idx))
keep = fbp.subset_keep(SUBSETS[HEADLINE])
pp, tp = [], []
for s in range(0, nprof, BATCH):
    sl = slice(s, min(s + BATCH, nprof))
    maps = fbp.generate(fm, ns, conds[sl], lss[sl], obs_norm[sl], keep, device, N_STEPS, seed=7 + s)
    for k in range(maps.shape[0]):
        i = s + k; d = np.load(files[idx[i]])
        tmaps = np.concatenate([d['target'], np.stack([dk for dk in (d[q] for q in THERMO_KEYS)])])
        pp.append(fbp.aperture_fb_profile(maps[k], M200[i], radii))
        tp.append(fbp.aperture_fb_profile(tmaps, M200[i], radii))
pp, tp = np.array(pp), np.array(tp)
fig, ax = plt.subplots(figsize=(6.5, 4.4))
ax.plot(radii, np.median(tp, 0), 'k-', label='truth')
ax.fill_between(radii, *np.percentile(tp, [16, 84], 0), color='k', alpha=0.15)
ax.plot(radii, np.median(pp, 0), 'r--', label=f'BIND ({HEADLINE})')
ax.fill_between(radii, *np.percentile(pp, [16, 84], 0), color='r', alpha=0.15)
ax.axvline(np.median(r200s), ls=':', color='gray', lw=1, label='median R200')
ax.set_xlabel('r [Mpc/h]'); ax.set_ylabel('local f_b(<r) = baryon/total'); ax.legend(fontsize=8)
ax.set_title('Predicted vs truth baryon-fraction profile'); plt.tight_layout(); plt.show()

## Summary & next (Stage 2)

Read the ablation/DMO-swap numbers: the headline `M+Y+Tx` predictor should beat `M`-only, and the DMO-marginal scatter (real-data proxy) should stay close to the self-DMO scatter with ~0.68 coverage. If so, the eROSITA-group application is well-founded.

**Stage 2 (eROSITA):** forward-model an eROSITA-style measurement (`Y`, `Tx`, lensing `M`) on the training sims to calibrate survey → BIND-observable converters, then ingest an eROSITA group catalog, predict f_b–M via `fbp.predict_fb_marginal` over matched-mass DMO templates, and compare to the catalog's independent f_gas–M. Reuse `fb_predict.py` throughout.